# Text Classification using Machine Learning Models
### Trump Tweet Sentiment Classification

## Step 1: Install & Import Required Libraries

In [1]:
import pandas as pd
import numpy as np
import re
import nltk

# Download required NLTK resources
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer, PorterStemmer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

stop_words = set(stopwords.words('english'))
wordnet_lemmatizer = WordNetLemmatizer()
porter_stemmer = PorterStemmer()

print("All libraries imported successfully!")

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


All libraries imported successfully!


## Step 2: Helper Functions for Text Cleaning

In [2]:
def lower_order(text):
    """Convert text to lowercase."""
    return text.lower()


def remove_urls(text):
    """Remove URLs from text."""
    return re.sub(r'http\S+|www\.\S+|https\S+', '', text)


def remove_emoji(text):
    """Remove emojis and unicode symbols."""
    emoji_pattern = re.compile(
        "["
        u"\U0001F600-\U0001F64F"
        u"\U0001F300-\U0001F5FF"
        u"\U0001F680-\U0001F6FF"
        u"\U0001F1E0-\U0001F1FF"
        u"\U00002700-\U000027BF"
        u"\U000024C2-\U0001F251"
        "]+",
        flags=re.UNICODE
    )
    return emoji_pattern.sub('', text)


def removeunwanted_characters(text):
    """Remove mentions, hashtags, RT marker, punctuation, and special chars."""
    text = re.sub(r'@\w+', '', text)           # Remove @mentions
    text = re.sub(r'#\w+', '', text)           # Remove #hashtags
    text = re.sub(r'RT\s+', '', text)          # Remove retweet tag
    text = re.sub(r'[^a-z\s]', '', text)       # Keep only letters & spaces
    text = re.sub(r'\s+', ' ', text).strip()   # Collapse extra whitespace
    return text


print("Helper functions ready!")

Helper functions ready!


## Step 3: Text Cleaning Pipeline

This function accepts a **single string** and applies all preprocessing steps:
lowercase → remove URLs → remove emojis → remove unwanted chars → tokenize → remove stopwords → lemmatize/stem

In [3]:
def text_cleaning_pipeline(dataset, rule="lemmatize"):
    """
    Full text preprocessing pipeline for a single input string.

    Parameters:
    -----------
    dataset : str
        Raw input text (e.g., a single tweet).
    rule : str
        Normalization strategy: 'lemmatize' (default) or 'stem'.

    Returns:
    --------
    str : Cleaned and normalized text as a single string.
    """
    # Step 1: Convert to lowercase
    data = lower_order(dataset)

    # Step 2: Remove URLs
    data = remove_urls(data)

    # Step 3: Remove emojis
    data = remove_emoji(data)

    # Step 4: Remove unwanted characters (mentions, hashtags, punctuation)
    data = removeunwanted_characters(data)

    # Step 5: Tokenize by splitting on whitespace
    tokens = data.split()

    # Step 6: Remove stopwords
    tokens = [word for word in tokens if word not in stop_words]

    # Step 7: Lemmatize or Stem
    if rule == "lemmatize":
        tokens = [wordnet_lemmatizer.lemmatize(word, pos='v') for word in tokens]
    elif rule == "stem":
        tokens = [porter_stemmer.stem(word) for word in tokens]
    else:
        print("Pick between lemmatize or stem")

    return " ".join(tokens)

## Step 4: Load Dataset

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
# Load the sentiment analysis dataset
# Expected columns: 'text' and 'label' (or 'Sentiment')
data = pd.read_csv("drive/MyDrive/AI and Machine Learning/workshops/workshop8/trum_tweet_sentiment_analysis.csv",
                   encoding="ISO-8859-1")

print("Dataset shape:", data.shape)
print("\nColumn names:", data.columns.tolist())
print("\nFirst 5 rows:")
data.head()

Dataset shape: (1850123, 2)

Column names: ['text', 'Sentiment']

First 5 rows:


,text,Sentiment
0,RT @JohnLeguizamo: #trump not draining swamp b...,0
1,ICYMI: Hackers Rig FM Radio Stations To Play A...,0
2,Trump protests: LGBTQ rally in New York https:...,1
3,"""Hi I'm Piers Morgan. David Beckham is awful b...",0
4,RT @GlennFranco68: Tech Firm Suing BuzzFeed fo...,0


In [6]:
# Check label distribution
print("Label distribution:")
print(data['Sentiment'].value_counts())

# Drop rows with missing text or labels
data = data.dropna(subset=['text', 'Sentiment'])
print(f"\nDataset size after dropping NaNs: {len(data)}")

Label distribution:
Sentiment
0    1244211
1     605912
Name: count, dtype: int64

Dataset size after dropping NaNs: 1850123


## Step 5: Apply Text Cleaning Pipeline

In [7]:
# Apply the cleaning pipeline to every tweet in the 'text' column
print("Cleaning text data... (this may take a moment)")
data['cleaned_text'] = data['text'].apply(
    lambda text: text_cleaning_pipeline(text, rule="lemmatize")
)

print("Done!")
print("\nBefore cleaning:")
print(data['text'][0])
print("\nAfter cleaning:")
print(data['cleaned_text'][0])

Cleaning text data... (this may take a moment)
Done!

Before cleaning:
RT @JohnLeguizamo: #trump not draining swamp but our taxpayer dollars on his trips to advertise his properties! @realDonaldTrumpÂ https://t.co/gFBvUkMX9z

After cleaning:
rt drain swamp taxpayer dollars trip advertise properties


## Step 6: Train-Test Split

In [8]:
X = data['cleaned_text']
y = data['Sentiment']

# Split 80% training, 20% testing; random_state ensures reproducibility
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y        # preserve class distribution in both splits
)

print(f"Training samples : {len(X_train)}")
print(f"Testing samples  : {len(X_test)}")

Training samples : 1480098
Testing samples  : 370025


## Step 7: TF-IDF Vectorization

TF-IDF (Term Frequency–Inverse Document Frequency) converts text into numerical feature vectors.
- **TF**: how often a word appears in a document
- **IDF**: penalizes words that appear in too many documents (less informative)

In [9]:
# Initialize TF-IDF Vectorizer
# max_features=5000: keeps the 5000 most frequent terms
# ngram_range=(1,2): includes unigrams and bigrams for richer features
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))

# Fit on training data and transform both splits
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)

print("TF-IDF feature matrix shape (train):", X_train_tfidf.shape)
print("TF-IDF feature matrix shape (test) :", X_test_tfidf.shape)

TF-IDF feature matrix shape (train): (1480098, 5000)
TF-IDF feature matrix shape (test) : (370025, 5000)


## Step 8: Train Logistic Regression Model

In [10]:
# Initialise and train Logistic Regression
# max_iter=1000 ensures convergence on larger datasets
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_tfidf, y_train)

print("Model trained successfully!")

Model trained successfully!


## Step 9: Evaluate the Model

In [11]:
# Generate predictions on the test set
y_pred = model.predict(X_test_tfidf)

# Overall accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Test Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")

print("\n--- Classification Report ---")
print(classification_report(y_test, y_pred))

Test Accuracy: 0.9053 (90.53%)

--- Classification Report ---
              precision    recall  f1-score   support

           0       0.92      0.94      0.93    248842
           1       0.88      0.83      0.85    121183

    accuracy                           0.91    370025
   macro avg       0.90      0.88      0.89    370025
weighted avg       0.90      0.91      0.90    370025



## Step 10: Try a Custom Prediction

In [12]:
def predict_sentiment(text):
    """Clean a raw tweet and predict its sentiment label."""
    cleaned = text_cleaning_pipeline(text)
    vec = tfidf.transform([cleaned])
    prediction = model.predict(vec)[0]
    label_map = {0: "Negative", 1: "Positive"}  # adjust if labels differ
    return label_map.get(prediction, str(prediction))


# Example predictions
tweets = [
    "This is the greatest country in the world! #MAGA",
    "Terrible policy, completely wrong direction for America.",
    "Looking forward to the new trade deal with China!"
]

for tweet in tweets:
    print(f"Tweet   : {tweet}")
    print(f"Sentiment: {predict_sentiment(tweet)}")
    print()

Tweet   : This is the greatest country in the world! #MAGA
Sentiment: Positive

Tweet   : Terrible policy, completely wrong direction for America.
Sentiment: Negative

Tweet   : Looking forward to the new trade deal with China!
Sentiment: Positive

